In [19]:
import pickle
with open("pre_split_data.pkl", "rb") as f:
    X, y, categorical, numerical = pickle.load(f)

In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score


In [21]:

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# One hot encode categorical features
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

X_train_ohe = pd.DataFrame(ohe.fit_transform(X_train[categorical]),
                           columns=ohe.get_feature_names_out(categorical),
                           index=X_train.index)

X_train_lin = pd.concat([X_train[numerical], X_train_ohe], axis=1)

X_test_ohe = pd.DataFrame(ohe.transform(X_test[categorical]),
                          columns=ohe.get_feature_names_out(categorical),
                          index=X_test.index)

X_test_lin = pd.concat([X_test[numerical], X_test_ohe], axis=1)

# Scale numerical features
scaler = StandardScaler()
X_train_lin[numerical] = scaler.fit_transform(X_train_lin[numerical])
X_test_lin[numerical] = scaler.transform(X_test_lin[numerical])

# Train logistic regression
LogReg = LogisticRegression(max_iter=1000, solver='saga') # saga is better for large dataset
LogReg.fit(X_train_lin, y_train)

y_pred_lr = LogReg.predict(X_test_lin)
y_prob_lr = LogReg.predict_proba(X_test_lin)[:,1]


c:\Users\sarah\anaconda3\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [23]:

print("=== Logistic Regression Baseline ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("ROC AUC:", roc_auc_score(y_test, y_prob_lr))
print("F1 score:", f1_score(y_test, y_pred_lr, pos_label='50000+.'))

=== Logistic Regression Baseline ===
Accuracy: 0.9524949693064011
ROC AUC: 0.9440308828659655
F1 score: 0.5121632226000523
